In [ ]:
!pip install segmentation_models_pytorch -q
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
import torch

def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split

class SUIMDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

        self.color_map = { # colors for each class segmentation(random)
            (0, 0, 0): 0,
            (0, 0, 255): 1,
            (0, 255, 0): 2,
            (255, 0, 0): 3,
            (255, 0, 255): 4,
            (255, 255, 0): 5,
            (255, 255, 255): 6,
            (0, 255, 255): 7
        }

    def __len__(self):
        return len(self.image_paths)

    def rgb_to_mask(self, rgb_image):
        np_img = np.array(rgb_image)
        mask = np.zeros(np_img.shape[:2], dtype=np.uint8)

        for color, idx in self.color_map.items():
            matches = np.all(np.abs(np_img - color) < 20, axis=-1)
            mask[matches] = idx

        return torch.from_numpy(mask).long()

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("RGB")

        if self.transform:
            image = self.transform(image)
            resize = transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST)
            mask = resize(mask)

        mask_tensor = self.rgb_to_mask(mask)

        return image, mask_tensor

image_dir = None
mask_dir = None

for root, dirs, files in os.walk(path):
    if "images" in dirs and "masks" in dirs:
        image_dir = os.path.join(root, "images")
        mask_dir = os.path.join(root, "masks")
        break

if not image_dir:
    print("Standard structure not found, searching recursively...")
    image_paths_raw = []
    mask_paths_raw = []
    for root, dirs, files in os.walk(path):
        for f in files:
            if f.endswith(('.jpg', '.bmp', '.png')):
                if "mask" in root.lower() or "mask" in f.lower():
                    mask_paths_raw.append(os.path.join(root, f))
                elif "image" in root.lower() or "img" in f.lower():
                    image_paths_raw.append(os.path.join(root, f))
    image_paths_raw.sort()
    mask_paths_raw.sort()
    all_image_paths = image_paths_raw
    all_mask_paths = mask_paths_raw
else:
    image_files = sorted(os.listdir(image_dir))
    mask_files = sorted(os.listdir(mask_dir))
    all_image_paths = [os.path.join(image_dir, f) for f in image_files]
    all_mask_paths = [os.path.join(mask_dir, f) for f in mask_files]

print(f"Found {len(all_image_paths)} images and {len(all_mask_paths)} masks.")

train_imgs, val_imgs, train_masks, val_masks = train_test_split(
    all_image_paths, all_mask_paths, test_size=0.2, random_state=42
)

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = SUIMDataset(train_imgs, train_masks, transform=transform)
val_dataset = SUIMDataset(val_imgs, val_masks, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

In [ ]:
# TO DO
import segmentation_models_pytorch as smp


model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=8
)

print("Model created.")

In [ ]:
# TO DO
from tqdm import tqdm

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for images, masks in tqdm(loader):
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for images, masks in tqdm(loader):
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            running_loss += loss.item()

    return running_loss / len(loader)

In [ ]:
# TO DO
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 10
train_losses = []
val_losses = []

print("Starting Training...")
for epoch in range(epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.title('Training and Valdation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
# TO DO
def visualize_predictions(model, loader, device, num_samples=3):
    model.eval()
    images, masks = next(iter(loader))
    images, masks = images.to(device), masks.to(device)

    with torch.no_grad():
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

    images = images.cpu()
    masks = masks.cpu()
    preds = preds.cpu()

    plt.figure(figsize=(12, 4 * num_samples))
    for i in range(num_samples):
        plt.subplot(num_samples, 3, i*3 + 1)
        img = images[i].permute(1, 2, 0).numpy()
        img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img = np.clip(img, 0, 1)
        plt.imshow(img)
        plt.title("Original Image")
        plt.axis("off")

        plt.subplot(num_samples, 3, i*3 + 2)
        plt.imshow(masks[i], cmap='jet', vmin=0, vmax=7)
        plt.title("Ground Truth")
        plt.axis("off")

        plt.subplot(num_samples, 3, i*3 + 3)
        plt.imshow(preds[i], cmap='jet', vmin=0, vmax=7)
        plt.title("Prediction")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

visualize_predictions(model, val_loader, device)